# M1 · physical — arm baseline

The real **RealMan RM75-6F** arm. Simulation twin: [`../simulation/submodule_0.py`](../simulation/submodule_0.py)
uses an RM65 stand-in model for the pick demo — this notebook talks to the real RM75-6F instead.

**Stack:** `Robotic_Arm.rm_robot_interface` (RealMan's official RM_API2 Ethernet binding) wrapped
behind `airo_robots.manipulators.PositionManipulator` — the same role `rtde_control`/`rtde_receive`
play for airo-robots' UR implementation. Backend code: [`real_arm.py`](real_arm.py).

This is a baseline: connect, read state, move in joint space, move in a straight line, servo a
short trajectory, disconnect. Nothing here is task-specific — M1's grasping submodules build on
top of these primitives.

### Before you move the arm
- Clear workspace: nothing within the arm's reach, no cables in its path.
- Speed ratios start low (`SPEED_RATIO = 10`, i.e. 10% of max joint/linear velocity).
- Keep a hand near the physical e-stop / teach pendant while running motion cells.
- **Interrupting the kernel does not stop the arm mid-trajectory.** Use the e-stop, or the
  `rm_set_arm_stop()` call in the troubleshooting section below.

### Setup on the robot PC
```bash
pip install Robotic_Arm
```


In [ ]:
import sys
from pathlib import Path

_here = [Path.cwd(), *Path.cwd().parents]
_candidates = [*_here, *(p / "src" / name for p in _here for name in ("m1", "M1"))]
M1_DIR = next((p for p in _candidates if (p / "physical" / "real_arm.py").exists()), None)
if M1_DIR is None:
    raise RuntimeError(f"could not find src/m1 from {Path.cwd()}; open this notebook from its own folder")
if str(M1_DIR / "physical") not in sys.path:
    sys.path.insert(0, str(M1_DIR / "physical"))

import numpy as np

from real_arm import RMArm

IP_ADDRESS = "192.168.1.18"  # controller IP, see the teach pendant / network settings
PORT = 8080
SPEED_RATIO = 10             # 1..100, used below as a fraction of the arm's max speeds


## Connect

Creates the RM_API2 connection (triple-thread mode) and reads the arm's current state once so
you can sanity-check the joint/pose values against the teach pendant before commanding anything.

In [ ]:
arm = RMArm(IP_ADDRESS, PORT)

joint_speed = SPEED_RATIO / 100 * min(arm.manipulator_specs.max_joint_speeds)
linear_speed = SPEED_RATIO / 100 * arm.manipulator_specs.max_linear_speed

print("joint configuration (rad):", arm.get_joint_configuration())
print("joint configuration (deg):", np.degrees(arm.get_joint_configuration()))
print("tcp pose:\n", arm.get_tcp_pose())


## Joint-space move

Moves 10 degrees away from the current configuration on the wrist-most joint (small, low-risk
motion regardless of current pose) and back. Check the printed configuration against the
pendant, then confirm at the prompt before it actually moves.

In [ ]:
q_start = arm.get_joint_configuration()
q_target = q_start.copy()
q_target[-1] += np.radians(10)

input(f"about to move joint 7 by +10 deg (from {np.degrees(q_start)} to {np.degrees(q_target)} deg), press enter to continue")
arm.move_to_joint_configuration(q_target, joint_speed=joint_speed).wait()
print("reached:", np.degrees(arm.get_joint_configuration()))

input("press enter to move back")
arm.move_to_joint_configuration(q_start, joint_speed=joint_speed).wait()
print("reached:", np.degrees(arm.get_joint_configuration()))


## Linear TCP move

Straight-line move 5 cm straight up from the current TCP pose and back, via `move_linear_to_tcp_pose`
(RM_API2 `rm_movel`).

In [ ]:
X_start = arm.get_tcp_pose()
X_up = X_start.copy()
X_up[2, 3] += 0.05

input(f"about to move 5cm up in a straight line (from z={X_start[2,3]:.3f} to z={X_up[2,3]:.3f}), press enter to continue")
arm.move_linear_to_tcp_pose(X_up, linear_speed=linear_speed).wait()
print("reached:\n", arm.get_tcp_pose())

input("press enter to move back down")
arm.move_linear_to_tcp_pose(X_start, linear_speed=linear_speed).wait()
print("reached:\n", arm.get_tcp_pose())


## Servo demo

Short (2s) zig-zag in TCP space using `servo_to_tcp_pose` (RM_API2 `rm_movep_canfd`, CANFD
streaming) instead of planned moves — this is the primitive higher-frequency control (e.g. visual
servoing) would use instead of `move_linear_to_tcp_pose`.

In [ ]:
control_freq = 50
amplitude = 0.02   # m
direction = np.array([1.0, 0.0, 0.0])

input("about to servo a 2s zig-zag of +-2cm along X, press enter to continue")
X_center = arm.get_tcp_pose()
for i in range(2 * control_freq):
    offset = amplitude * np.sin(2 * np.pi * (i / control_freq))
    X_servo = X_center.copy()
    X_servo[:3, 3] += offset * direction
    arm.servo_to_tcp_pose(X_servo, 1 / control_freq).wait()

arm.move_linear_to_tcp_pose(X_center, linear_speed=linear_speed).wait()
print("reached:\n", arm.get_tcp_pose())


## Shutdown

Releases the RM_API2 connection so the next process (or the teach pendant) can claim it.

In [ ]:
arm.close()


## Next

[`real_arm.py`](real_arm.py) — the `PositionManipulator` implementation used above; run it directly
(`python real_arm.py --ip_address ...`) for the full `airo_robots` manual test suite
(IK/FK, moves, servo).

## Troubleshooting

| Symptom | Cause |
|---|---|
| `Robotic_Arm not installed` | `pip install Robotic_Arm` in the `int2026` env (robot PC). |
| `Could not connect to the RM75-6F` | Wrong IP/port, arm not powered, not on the same subnet, or the teach pendant / another client already holds the connection. |
| `rm_movej`/`rm_movel` returns non-zero | Check the RM_API2 error code against [the appendix](https://develop.realman-robotics.com/en/robot/apierrorList2/) — commonly an unreachable pose or joint limit violation. |
| Arm won't stop after kernel interrupt | Use the physical e-stop, or call `arm.arm.rm_set_arm_stop()` from a fresh cell/kernel. |
| Motion looks right but `.wait()` times out | The L2 thresholds in `real_arm.py` (`_pose_reached_L2_threshold` / `_joint_config_reached_L2_threshold`) may be tighter than the arm's settling behavior — loosen them or increase the timeout. |
